In [2]:
pip install pandas requests tqdm pyarrow chardet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import io
import re
import csv
import sys
import time
import zipfile
import string
import typing as T
from datetime import datetime, timedelta, timezone

import requests
import pandas as pd
from tqdm import tqdm

# --------------------------
# USER SETTINGS
# --------------------------
START_UTC = "2024-01-01 00:00"   # UTC!
END_UTC   = "2024-01-07 00:00"   # UTC!
OUT_PATH  = "gdelt_ai_cloud_datacenters_sentiment.parquet"  # .csv or .parquet supported
WRITE_MODE = "parquet"  # "parquet" or "csv"

# Your domain keywords (free text & theme tokens).
# We'll match BOTH: (a) human words in URL/Source and (b) GKG V2Themes-style tokens (UPPER_WITH_UNDERSCORES; semicolon-separated).
HUMAN_PATTERNS = [
    r"data\s?cent(er|re)s?",      # data center/centre(s)
    r"\bserver(s)?\b",
    r"\bcloud\b",
    r"\bartificial intelligence\b",
    r"\bAI\b"
]
# V2Themes-ish tokens you likely care about (expand as needed).
THEME_TOKENS = [
    "ARTIFICIAL_INTELLIGENCE",
    "MACHINE_LEARNING",
    "CLOUD_COMPUTING",
    "DATA_CENTER",
    "DATA_CENTERS",
    "DATACENTER",
    "SERVER_FARMS",
    "SERVER",
    "COLOCATION",
    "HYPERSCALE"
]

# Network settings
TIMEOUT = 30
RETRIES = 3
SLEEP_BETWEEN = 0.25  # seconds between files to be polite

# --------------------------
# Helpers
# --------------------------
BASE = "http://data.gdeltproject.org/gdeltv2/{stamp}.gkg.csv.zip"  # raw files. :contentReference[oaicite:1]{index=1}

def iter_15min(start: datetime, end: datetime):
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(minutes=15)

def ts_stamp(dt: datetime) -> str:
    # GDELT stamp is YYYYMMDDHHMMSS with seconds always 00
    return dt.strftime("%Y%m%d%H%M%S")

def fetch_bytes(url: str) -> bytes:
    for attempt in range(1, RETRIES+1):
        try:
            r = requests.get(url, timeout=TIMEOUT)
            if r.status_code == 200:
                return r.content
            # 404s are common for missing windows — just return b""
            if r.status_code == 404:
                return b""
        except requests.RequestException:
            if attempt == RETRIES:
                raise
        time.sleep(0.8 * attempt)
    return b""

# V2Tone looks like "tone,positive,negative,polarity,activityrefdensity,grouprefdensity"
TONE_REGEX = re.compile(
    r"^-?\d+(?:\.\d+)?(?:,-?\d+(?:\.\d+)?){5}$"
)

# Build regexes for filtering
HUMAN_REGEX = re.compile("|".join(HUMAN_PATTERNS), re.IGNORECASE)
THEME_SET = set(THEME_TOKENS)

def parse_zip_csv(zip_bytes: bytes) -> T.Iterable[T.List[str]]:
    if not zip_bytes:
        return []
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        # find first CSV inside the ZIP
        name = [n for n in zf.namelist() if n.endswith(".csv")][0]
        with zf.open(name) as fh:
            # GKG v2 is tab-delimited and may contain large rows; use csv module
            text = io.TextIOWrapper(fh, encoding="utf-8", errors="replace", newline="")
            reader = csv.reader(text, delimiter="\t")
            for row in reader:
                # skip empty
                if not row or all(c == "" for c in row):
                    continue
                yield row

def extract_fields(row: T.List[str]) -> dict:
    """
    We don't assume fixed positions (schema can shift when fields are empty).
    Strategy:
      - DATE: usually col 1 (YYYYMMDDHHMMSS)
      - SourceCommonName: often col 3
      - DocumentIdentifier (URL): often col 4 or 5
      - V2Tone: detect by TONE_REGEX
      - V2Themes / V2EnhancedThemes: those with semicolon-delimited UPPER tokens
    We'll be defensive and search across columns.
    """
    out = {
        "DATE": None,
        "SourceCommonName": None,
        "DocumentIdentifier": None,
        "V2ToneStr": None,
        "V2ThemesGuess": None,
        "V2EnhancedThemesGuess": None
    }

    # try DATE at index 1
    if len(row) > 1 and re.fullmatch(r"\d{14}", row[1] or ""):
        out["DATE"] = row[1]

    # try SourceCommonName at idx 3 if looks like a hostname-ish to
